# Player identity check: `Joey` vs `joeyLiu`

**Question:** my plays show up as both `Joey` and `joeyLiu`. Does the weekly GitHub Actions build (`buildSite.py`) count them all as me?

`buildSite.py` decides who "me" is with:

```python
def is_me(p):
    return p['username'].lower() == USERNAME.lower() or p['name'].lower() == ME.lower()
```

with `USERNAME='joeyLiu'` and `ME='Joey'`. Every player entry on a BGG play has two fields, a **name** (free text) and a **username** (set only when the player is linked to a BGG account). So `is_me` matches when the username is `joeyLiu` **or** the name is `Joey`. This notebook pulls the real play data and checks for entries that slip through.

In [1]:
import os, re
from collections import Counter

# Load BGG_TOKEN from set_env.ps1 (gitignored), the same way the local scripts do
if not os.environ.get('BGG_TOKEN') and os.path.exists('set_env.ps1'):
    m = re.search(r'\$env:BGG_TOKEN\s*=\s*"([^"]+)"', open('set_env.ps1').read())
    if m:
        os.environ['BGG_TOKEN'] = m.group(1)
os.environ.setdefault('BGG_CACHE_DIR', '.bgg_cache')  # gitignored; makes reruns instant

import pandas as pd
import buildSite
from bgg_api import get_plays

print('USERNAME =', buildSite.USERNAME, '| ME =', buildSite.ME, '| PARTNER =', buildSite.PARTNER)
plays = get_plays(buildSite.USERNAME)
print(len(plays), 'logged plays,', sum(p['qty'] for p in plays), 'counting quantity')

USERNAME = joeyLiu | ME = Joey | PARTNER = Mickey


525 logged plays, 525 counting quantity


## 1. Every (name, username) combination that looks like Joey
One row per distinct player identity on the plays, with whether `is_me` currently matches it.

In [2]:
rows = [{'name': x['name'], 'username': x['username'], 'qty': p['qty']} for p in plays for x in p['players']]
players = pd.DataFrame(rows)
ident = (players.groupby(['name', 'username'], dropna=False)
         .agg(entries=('qty', 'size'), plays=('qty', 'sum')).reset_index())
ident['is_me'] = [buildSite.is_me({'name': n, 'username': u}) for n, u in zip(ident.name, ident.username)]
joeyish = ident[ident.name.str.contains('joey|liu', case=False) | ident.username.str.contains('joey|liu', case=False)]
joeyish.sort_values('plays', ascending=False)

,name,username,entries,plays,is_me
24,Joey,,511,511,True
25,Joey,joeyLiu,10,10,True


## 2. Plays where I'm *not* detected at all
I log these plays myself, so I should be at the table on almost all of them. A play with no `is_me` player is a likely miss, unless it's a play I logged for someone else.

In [3]:
missing = [p for p in plays if not any(buildSite.is_me(x) for x in p['players'])]
print(len(missing), 'of', len(plays), 'plays have no player matched as me')
pd.DataFrame([{'date': p['date'], 'game': p['game'],
               'players': ', '.join(f"{x['name']}" + (f" [{x['username']}]" if x['username'] else '') for x in p['players'])}
              for p in missing]).head(40)

4 of 525 plays have no player matched as me


,date,game,players
0,2025-06-19,Railroad Ink: Deep Blue Edition,
1,2023-12-26,Kingdomino,"Mickey, mom"
2,2023-12-26,Patchwork,"Mickey, mom"
3,2023-12-25,The Crew: The Quest for Planet Nine,"Mickey, dad, dave, emily, matt"


## 3. Who appears on those unmatched plays?
If there's a `Joey` variant we haven't caught, it'll stand out here.

In [4]:
Counter((x['name'], x['username']) for p in missing for x in p['players']).most_common(20)

[(('Mickey', ''), 3),
 (('mom', ''), 2),
 (('dad', ''), 1),
 (('dave', ''), 1),
 (('emily', ''), 1),
 (('matt', ''), 1)]

## 4. Plays where I'm matched *twice*
If `Joey` and a `joeyLiu`-linked entry both sit on one play, `find()` takes the first one and the second gets counted as an opponent in the "other people" table.

In [5]:
double = [p for p in plays if sum(buildSite.is_me(x) for x in p['players']) > 1]
print(len(double), 'plays with more than one is_me player')
pd.DataFrame([{'date': p['date'], 'game': p['game'], 'players': [(x['name'], x['username']) for x in p['players']]} for p in double]).head(20)

0 plays with more than one is_me player


""


## 5. Does a Joey variant leak into the opponents table?
This rebuilds the same `opponents` list the site shows and looks for Joey-like names.

In [6]:
os.environ.setdefault('BGG_CACHE_DIR', '.bgg_cache')
data = buildSite.build()
opp = pd.DataFrame(data['opponents'])
print('Joey-like opponents:', opp[opp.name.str.contains('joey|liu', case=False)].to_dict('records'))
opp

Fetching collection for joeyLiu
Fetching plays for joeyLiu


Fetching details for 197 games


Joey-like opponents: []


,name,plays,me,them,last
0,dad,63,11,15,2026-04-15
1,samuel,62,18,13,2025-12-20
2,matt,51,13,14,2026-01-03
3,mom,42,9,5,2026-04-15
4,peter,36,11,6,2026-01-19
5,mitchell,19,8,2,2026-01-19
6,sophie,16,4,3,2026-01-01
7,Abby,10,2,3,2025-07-19
8,Eli,10,2,1,2025-07-19
9,Danielle,8,0,1,2026-02-01


## Conclusion (run on 2026-11-25)

- **Every Joey entry is already combined.** On all 520 plays where I appear, the player name is `Joey`. On 511 of them the player isn't linked to a BGG account, and on 9 it's linked to `joeyLiu`. `is_me` matches both forms (the name alone is enough), so they count as the same person.
- **No `joeyLiu` player name exists.** "joeyLiu" only shows up as the username on the account link, never as a separate name.
- **No double-counting.** No play has two "me" entries, and no Joey variant leaks into the opponents table.
- **The 4 unmatched plays are real:** 3 holiday games where Mickey played without me, plus 1 Railroad Ink play logged with no players at all.
- **GitHub Actions uses the same defaults.** `docs.yaml` doesn't override `BGG_USERNAME` or `ME_NAME`, so the weekly build runs exactly this logic.

Re-run this notebook if the logging habits change (for example, adding myself as `Joey L` or `JoeyLiu` by name).

## 6. Follow-up: recent Ticket to Ride plays missing from the site

BG Stats shows recent Ticket to Ride plays, but they aren't in the site's play log. First question: is BGG itself missing them, or is the site dropping them? The site can only show plays that are logged on the BGG account `joeyLiu`. (BGG's plays API returns only the plays that account *logged*, not plays where someone else tagged it.)

In [7]:
ttr = [(p['date'], p['game'], [x['name'] + (f" [{x['username']}]" if x['username'] else '') for x in p['players']])
       for p in plays if 'ticket to ride' in p['game'].lower()]
pd.DataFrame(ttr, columns=['date', 'game', 'players']).head(10)

,date,game,players
0,2026-11-15,Ticket to Ride: Nordic Countries,"[Mickey, Joey [joeyLiu]]"
1,2026-04-15,Ticket to Ride Map Collection 1: Asia + Legend...,"[Joey, Mickey, dad, mom]"
2,2025-07-19,Ticket to Ride Map Collection 5: United Kingdo...,"[Abby, Eli, Gabriel, Joey, peter]"
3,2025-05-26,Ticket to Ride: Europe,"[Joey, Mickey, Phil]"
4,2025-02-17,Ticket to Ride Map Collection 1: Asia + Legend...,"[samuel, peter, Joey, Mickey]"
5,2024-12-30,Ticket to Ride Map Collection 1: Asia + Legend...,"[dad, mom, Joey, Mickey]"
6,2024-12-23,Ticket to Ride Map Collection 1: Asia + Legend...,"[Joey, Mickey, mom]"
7,2024-11-30,Ticket to Ride Map Collection 1: Asia + Legend...,"[Joey, Mickey, mom]"
8,2024-07-28,Ticket to Ride Map Collection 1: Asia + Legend...,"[Joey, Mickey, Anonymous player, Anonymous pla..."
9,2024-06-15,Ticket to Ride Map Collection 1: Asia + Legend...,"[Joey, Mickey, samuel]"


Bypass the local cache and ask BGG directly what's been logged since September:

In [8]:
from bgg_api import _soup
import bgg_api
saved, os.environ['BGG_CACHE_DIR'] = os.environ.get('BGG_CACHE_DIR'), ''
fresh = _soup(bgg_api.fetch('https://boardgamegeek.com/xmlapi2/plays?username=joeyLiu&mindate=2026-09-01'))
os.environ['BGG_CACHE_DIR'] = saved
print('plays on BGG since 2026-09-01:', fresh.find('plays')['total'])
[(p['date'], p.find('item')['name']) for p in fresh.find_all('play')]

plays on BGG since 2026-09-01: 5


[('2026-11-15', 'Ticket to Ride: Nordic Countries'),
 ('2026-10-31', 'Betrayal at House on the Hill'),
 ('2026-10-31', 'Betrayal at House on the Hill'),
 ('2026-10-25', 'Ark Nova'),
 ('2026-10-25', 'Res Arcana')]

**Result:** BGG has no Ticket to Ride play after **2026-04-15**, and no play of any kind after **2026-10-31**. The live site was rebuilt on 2026-11-23, and its latest play is also 2026-10-31. So the site is up to date with BGG. **The recent plays never reached the `joeyLiu` BGG account.** That points to the BG Stats → BGG sync, not to `buildSite.py`.

## 7. After fixing the BG Stats sync (2026-11-25)

After re-syncing from BG Stats, a new play appeared on BGG: **2026-11-15 Ticket to Ride: Nordic Countries**. In that play BG Stats sent my player entry as **name = `Joey ( joeyLiu)`** with **no username attached**. This is the `joey` / `joeyLiu` split from the app, and the original `is_me` didn't match it, so I'd have shown up as an opponent named "Joey ( joeyLiu)" and the play would have been left out of the Mickey rivalry stats.

**Fix (in `bgg_api.get_plays`):** when a player has no username and the name ends in `(something)`, split it into name `Joey` and username `joeyLiu`. Every page then sees a normal "Joey / joeyLiu" player.

In [9]:
from bgg_api import _NAME_WITH_USER
for raw in ['Joey ( joeyLiu)', 'Joey (joeyLiu)', 'Joey', 'Mickey']:
    m = _NAME_WITH_USER.match(raw)
    print(f'{raw!r:20} ->', (m.group(1), m.group(2)) if m else (raw, ''))

'Joey ( joeyLiu)'    -> ('Joey', 'joeyLiu')
'Joey (joeyLiu)'     -> ('Joey', 'joeyLiu')
'Joey'               -> ('Joey', '')
'Mickey'             -> ('Mickey', '')


In [10]:
for p in plays:
    if p['date'] >= '2026-11-01':
        print(p['date'], p['game'], [(x['name'], x['username'], 'me' if buildSite.is_me(x) else '') for x in p['players']])
print('names still containing "(":', sorted({x['name'] for p in plays for x in p['players'] if '(' in x['name']}))
print('rivalry plays with Mickey:', data['rivalry']['together'], '| latest:', data['rivalry']['log'][-1])

2026-11-15 Ticket to Ride: Nordic Countries [('Mickey', '', ''), ('Joey', 'joeyLiu', 'me')]
names still containing "(": []
rivalry plays with Mickey: 445 | latest: ['2026-11-15', 2, 'partner', 1]
